In [138]:
import bs4
import curl_cffi

In [139]:
def parse_ul(soup: bs4.BeautifulSoup) -> str:
    list_items = soup.find_all('li')
    list_items = [li.text.strip() for li in list_items]
    list_items = [f'- {li}' for li in list_items]
    return "\n".join(list_items)

In [146]:
def parse_ol(soup: bs4.BeautifulSoup) -> str:
    list_items = soup.find_all('li')
    list_items = [li.text.strip() for li in list_items]
    list_items = [f'{i+1}. {li}' for i, li in enumerate(list_items)]
    return "\n".join(list_items)

In [152]:
def chunkify(content: bs4.element.ResultSet) -> list[list[bs4.element.Tag]]:
    bs4_chunks = []
    i = 0
    while i < len(content):
        element = content[i]
        if element.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']:
            chunk = [element]
            i += 1
            while i < len(content) and content[i].name not in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']:
                chunk.append(content[i])
                i += 1
            bs4_chunks.append(chunk)
        else:
            i += 1  # Skip non-heading elements or handle them separately
    return bs4_chunks  # Fix: Return the correct variable

In [157]:
def prettify_chunks(chunks: list[list[bs4.element.Tag]]) -> list[str]:
    str_chunks = []  # Initialize list for strings
    for chunk in chunks:
        processed = []
        for element in chunk:
            if element.name == 'ul':
                processed.append(parse_ul(element))
            elif element.name == 'ol':
                processed.append(parse_ol(element))
            else:
                processed.append(element.text.strip())
        str_chunks.append("\n\n".join(processed))
    return str_chunks

In [158]:
def scrape(url: str) -> list[str]:
    file = curl_cffi.get(url, impersonate='chrome')
    soup = bs4.BeautifulSoup(file.text, 'html.parser')
    content = soup.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'ul', 'ol'])

    content = chunkify(content)
    content = prettify_chunks(content)
    
    return content

In [159]:
content = (scrape('https://www.parktool.com/en-us/blog/repair-help/rear-derailleur-advanced-troubleshooting?srsltid=AfmBOooF7O0cbpycR3GJ8pV5OcyOCho88Gi4mUVrCuFGe5UWXciC8LMV'))

In [160]:
for i in content:
    print(i)
    print("\n\n ---------------------------\n\n")

Shopping Cart



No items in cart.

Subtotal: $0.00



- New Products
- Tools
- Repair Stands
- Park Tool Gear
- E-Gift Cards
- Catalogs

- Repair Help
- Park Tool School
- Shop Talk
- Wheel Tension App

- Product Documentation
- Replacement Parts
- Warranty
- Services
- Order Resources

- About Us
- Advocacy
- Careers
- Sponsorship
- News
- Contact Us

- Sign In
- Register

- United States
- International

- Home
- /
- Repair Help
- /
- Rear Derailleur — Advanced Troubleshooting


 ---------------------------


Rear Derailleur — Advanced Troubleshooting

This article will review troubleshooting rear mechanical derailleur shifting.


 ---------------------------


Preliminary Info

- Rear Derailleur Adjustment - View Article

Troubleshooting refers to extra issues or features that go beyond normal set up and adjustment. It is best always to begin with basic adjustment procedures of limit screw settings, indexing, and B-screw adjustments. This will typically take care of most shifting p

In [164]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import PointStruct, VectorParams, Distance, QueryResponse
from ollama import Client as OllamaClient, ChatResponse
from typing import Mapping, Any, Optional, Sequence
import os
import hashlib

In [165]:
_ollama_client: Optional["OllamaClient"] = None

def _get_ollama_client() -> OllamaClient:
    global _ollama_client
    if _ollama_client is None:
        host = os.getenv("OLLAMA_HOST", "http://localhost:11434")
        _ollama_client = OllamaClient(host=host)
    return _ollama_client

In [166]:
_qdrant_client: Optional["QdrantClient"] = None

def _get_qdrant_client() -> QdrantClient:
    global _qdrant_client
    if _qdrant_client is None:
        url = os.getenv("QDRANT_HOST", "http://localhost:6333")
        _qdrant_client = QdrantClient(url=url)
    return _qdrant_client

In [169]:
def embed(text:str) -> list[float]:
    _ollama_client = _get_ollama_client()
    response: ChatResponse = _ollama_client.embed(model="nomic-embed-text", input=text) # type: ignore
    return response.embeddings[0] 

In [168]:
def qdrant_write(document: str, collection: str) -> None:
    vector: list[float] = embed(document)
    point = PointStruct(
        id=document_id(document), #automatically translated to UUID
        vector=vector,
        payload={"text": document}
        )
    
    qdrant_client = _get_qdrant_client()

    if qdrant_client.collection_exists(collection_name=collection):
        pass
    else:
        qdrant_client.create_collection(
            collection_name=collection,
            vectors_config=VectorParams(
                size=len(vector),
                distance=Distance.COSINE
            )
        )

    qdrant_client.upsert(
        collection_name=collection,
        points=[point],
    )

In [171]:
def document_id(document: str, truncate: int = 32) -> str:
    normalized = " ".join(document.split()).strip().lower()
    encoded = normalized.encode('utf-8')
    hash = hashlib.sha256()
    hash.update(encoded)
    return hash.hexdigest()[:truncate]

In [178]:
def search(vector: Sequence[float], collection_name: str, limit: int) -> Sequence[str]:
    qdrant_client = _get_qdrant_client()
    results: QueryResponse = qdrant_client.query_points(
        collection_name=collection_name,
        query=vector,
        limit=limit,
    )

    return [results.points[i].payload["text"] for i in range(len(results.points))]

In [172]:
for i in content:
    qdrant_write(i, collection="General")

In [174]:
_qdrant_client.scroll(collection_name="General", scroll_filter=None, limit=1)

([Record(id='002b94b6-e740-4972-113d-56ee6f737775', payload={'text': 'Chain Issues\n\nChains take a lot of abuse, transmitting the power to the rear wheel, and they don’t last forever. The internal parts of the chain, the rollers and rivets, begin to wear down and give the illusion of stretching. This wear can cause the chain to mesh poorly with the cogs and chainrings, causing poor shifting and premature wear to the cogs. Inspect your chain to see if it falls within the limits of tolerable wear.\n\nAlso check for adequate lubrication. A dry chain will feel sluggish when shifting and will make excessive noise when pedaling.'}, vector=None, shard_key=None, order_value=None)],
 '1d3f7a2e-b2ca-f6a2-a93b-3014e2b5b20c')

In [175]:
query = "What is wrong with my rear derailleur?"

In [176]:
vec_query = embed(query)

In [180]:
results = search(vec_query, collection_name="General", limit=5)

In [181]:
for i in results:
    print(i)
    print("\n\n ---------------------------\n\n")

Bent Hanger

The rear derailleur is attached to the frame with a hanger. These are commonly aluminum but may also be steel. Crashing or throwing the bike down on the right side can bend this hanger. Once bent, the derailleurs and pulleys are not going to align with the sprockets. Sight from behind the bike and see if the hanger looks parallel to the wheel and sprockets.

It is possible to both check and repair the hanger with a derailleur hanger alignment tool, such as the DAG-2.2 or DAG-3. See how to correct this in our Rear Derailleur Hanger Alignment article.


 ---------------------------


Worn Derailleur

All derailleurs wear out at some point. Check for wear by pulling laterally on the lower cage. Compare this movement in the linkage of a new derailleur. Sloppy pivot and linkages will produce inconsistent shifting, and the only solution is a new derailleur.


 ---------------------------


Rear Derailleur — Advanced Troubleshooting

This article will review troubleshooting rear 

In [182]:
def extract_and_store(url: str, collection: str) -> None:
    content = scrape(url)
    for i in content:
        qdrant_write(i, collection=collection)

In [183]:
extract_and_store('https://www.parktool.com/en-us/blog/repair-help/rear-derailleur-advanced-troubleshooting?srsltid=AfmBOooF7O0cbpycR3GJ8pV5OcyOCho88Gi4mUVrCuFGe5UWXciC8LMV', collection="General")

In [184]:
extract_and_store('https://www.sram.com/en/learn/axs-bike-care-and-maintenance', collection="SRAM")

In [185]:
_qdrant_client.scroll(collection_name="SRAM", scroll_filter=None, limit=10)

([Record(id='1290effe-aa4b-13e2-cbb4-d1895944a5e3', payload={'text': 'Company\n\n- About\n- Media\n- Careers\n- Logos\n- Locations\n- Legal Resources\n\n© 2026 SRAM LLC. All rights reserved.\n\n- Privacy Policy & Terms of Use\n- Accessibility Policy\n- Contrast\n\n- EN\n\n English\n Spanish\n\nChange Region\n- English\n- Spanish\n- Change Region\n\n- English\n- Spanish\n- Change Region'}, vector=None, shard_key=None, order_value=None),
  Record(id='1fde5b2a-f615-f3ec-91f5-c634e3795767', payload={'text': 'Remove Your Rear Wheel'}, vector=None, shard_key=None, order_value=None),
  Record(id='3cd02df0-a05f-42fc-e4ca-b3f4df4316fb', payload={'text': 'FREQUENTLY ASKED QUESTIONS\n\n\n\nLEARN MORE'}, vector=None, shard_key=None, order_value=None),
  Record(id='537e1049-e118-2c82-5959-ff8b1e1aab21', payload={'text': '13-SPEED XPLR AXS DERAILLEURS\n\nFor the ultimate robustness and rebuildability, the new RED XPLR AXS rear derailleur features more key replaceable parts than ever before. You can 